# Exploratory Data Analysis (EDA) - E-Commerce Sales Intelligence

This notebook conducts a thorough exploratory data analysis on cleaned e-commerce transaction data.
It answers key business questions around revenue, profit margins, product performance, regional growth, and customer purchasing patterns.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Load cleaned data
data_path = '../data/processed/cleaned_sales.csv'
if not os.path.exists(data_path):
    data_path = 'data/processed/cleaned_sales.csv'
    
df = pd.read_csv(data_path)
df['order_date'] = pd.to_datetime(df['order_date'])
print(f"Loaded {len(df)} cleaned records from {data_path}.")
df.head()

## 1. Dataset Overview & Summary Statistics

In [ ]:
print("=== DataFrame Info ===")
df.info()

print("\n=== Descriptive Statistics ===")
df[['quantity', 'unit_price', 'discount', 'cost', 'sales', 'profit', 'profit_margin']].describe().round(2)

## 2. Monthly Revenue & Profit Trends (Seasonality Analysis)
**Business Question**: Which months generate the highest revenue and profit?

In [ ]:
df['year_month'] = df['order_date'].dt.to_period('M')
monthly = df.groupby('year_month')[['sales', 'profit']].sum().reset_index()
monthly['year_month'] = monthly['year_month'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(monthly['year_month'], monthly['sales'], marker='o', color='#1f77b4', linewidth=2.5, label='Monthly Revenue ($)')
ax1.set_ylabel('Revenue ($)', color='#1f77b4', fontsize=12)
ax1.set_xticks(range(len(monthly['year_month'])))
ax1.set_xticklabels(monthly['year_month'], rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly['year_month'], monthly['profit'], marker='s', color='#2ca02c', linewidth=2.5, linestyle='--', label='Monthly Profit ($)')
ax2.set_ylabel('Profit ($)', color='#2ca02c', fontsize=12)

plt.title('Monthly Revenue & Profit Growth Trends', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 3. Product Category Performance
**Business Question**: Which categories generate the most revenue and profit?

In [ ]:
cat_summary = df.groupby('category').agg(
    total_revenue=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    avg_margin=('profit_margin', 'mean')
).reset_index().sort_values(by='total_revenue', ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].bar(cat_summary['category'], cat_summary['total_revenue'], color='#3498db')
ax[0].set_title('Total Revenue by Category', fontsize=13, fontweight='bold')
ax[0].set_ylabel('Revenue ($)')
ax[0].tick_params(axis='x', rotation=30)

ax[1].bar(cat_summary['category'], cat_summary['total_profit'], color='#2ecc71')
ax[1].set_title('Total Profit by Category', fontsize=13, fontweight='bold')
ax[1].set_ylabel('Profit ($)')
ax[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Regional Breakdown
**Business Question**: Which regions perform best?

In [ ]:
reg_summary = df.groupby('region').agg(total_revenue=('sales', 'sum')).reset_index()

plt.figure(figsize=(8, 8))
plt.pie(reg_summary['total_revenue'], labels=reg_summary['region'], autopct='%1.1f%%', colors=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#9b59b6'], startangle=140)
plt.title('Regional Revenue Share Contribution', fontsize=14, fontweight='bold')
plt.show()

## 5. Top 10 Best Selling Products
**Business Question**: Which specific products drive the majority of revenue?

In [ ]:
top_prods = df.groupby('product_name')['sales'].sum().reset_index().sort_values(by='sales', ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.barh(top_prods['product_name'][::-1], top_prods['sales'][::-1], color='#8e44ad')
plt.title('Top 10 Products by Total Revenue ($)', fontsize=14, fontweight='bold')
plt.xlabel('Revenue ($)')
plt.tight_layout()
plt.show()